In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('cleaned_political_bias_data (1).csv')
df.head(2)

,News_ID,Title,News_Body,Stance,Label,issue,topic,roundup
0,0,north carolina house race carries unlikely han...,charlotte no the closely watched special us ho...,lean left,liberal,illegal ballot harvesting prompted re do of no...,voting rights and voter fraud,charlotte no the closely watched special us ho...
1,0,trump unloads on disloyal democratic house can...,high stakes were matched by some of president ...,lean right,conservative,illegal ballot harvesting prompted re do of no...,voting rights and voter fraud,high stakes were matched by some of president ...


In [21]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

Device: cpu


In [3]:
!pip install transformers torch --quiet

In [4]:
!pip install --upgrade transformers tokenizers --quiet

In [5]:
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import XLNetTokenizer, XLNetModel

In [6]:
print(f"Total samples : {len(df)}")
print(df['Label'].value_counts())
print(df['Stance'].value_counts())

Total samples : 9190
Label
liberal         3066
conservative    3062
center          3062
Name: count, dtype: int64
Stance
center        3054
lean left     2237
lean right    2088
right          974
left           829
mixed            6
not rated        2
Name: count, dtype: int64


## Setting Up Hyperparameters

In [7]:
XLNET_MODEL_NAME = "xlnet-base-cased"
MAX_LENGTH = 64
PROJECTION_DIM = 256
BATCH_SIZE = 16

LABEL2ID = {
    "liberal" : 0,
    "center" : 1,
    "conservative" : 2,
}

## Setting up Stance to Ideology

In [8]:
stance_to_ideology = {
    "left" : "far left leaning ideology",
    'lean left' : 'slightly left leaning ideology',
    'center'    : 'centrist ideology with balanced reporting',
    'lean right': 'slightly right leaning ideology',
    'right'     : 'far right leaning ideology',
    'mixed'     : 'mixed or undefined ideological leaning',
    'not rated' : 'ideology not rated or unknown',
}

XLNet Data Pre-processing: Multi-Feature Tokenization Pipeline


In [13]:
class PoliticalBiasDataset(Dataset):
    def __init__(self, df, tokenizer, max_length):
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.issues     = df['issue'].fillna('unknown issue').tolist()
        self.topics     = df['topic'].fillna('unknown topic').tolist()
        self.ideologies = [
            stance_to_ideology.get(str(s).strip().lower(), 'ideology not rated or unknown')
            for s in df['Stance'].tolist()
        ]
        self.labels = [
            LABEL2ID.get(str(l).strip().lower(), 1)
            for l in df['Label'].tolist()
        ]

    def _tokenize(self, text):
        return self.tokenizer(
            text,
            max_length     = self.max_length,
            padding        = 'max_length',
            truncation     = True,
            return_tensors = 'pt',
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        issue_enc    = self._tokenize(self.issues[idx])
        topic_enc    = self._tokenize(self.topics[idx])
        ideology_enc = self._tokenize(self.ideologies[idx])
        return {
            'issue_input_ids'         : issue_enc['input_ids'].squeeze(0),
            'issue_attention_mask'    : issue_enc['attention_mask'].squeeze(0),
            'issue_token_type_ids'    : issue_enc.get('token_type_ids', torch.zeros(self.max_length, dtype=torch.long)).squeeze(0),
            'topic_input_ids'         : topic_enc['input_ids'].squeeze(0),
            'topic_attention_mask'    : topic_enc['attention_mask'].squeeze(0),
            'topic_token_type_ids'    : topic_enc.get('token_type_ids', torch.zeros(self.max_length, dtype=torch.long)).squeeze(0),
            'ideology_input_ids'      : ideology_enc['input_ids'].squeeze(0),
            'ideology_attention_mask' : ideology_enc['attention_mask'].squeeze(0),
            'ideology_token_type_ids' : ideology_enc.get('token_type_ids', torch.zeros(self.max_length, dtype=torch.long)).squeeze(0),
            'label'                   : torch.tensor(self.labels[idx], dtype=torch.long),
        }

Load Tokenizer

In [14]:
tokenizer = XLNetTokenizer.from_pretrained(XLNET_MODEL_NAME)

 Create DataLoader

In [15]:
dataset    = PoliticalBiasDataset(df, tokenizer, MAX_LENGTH)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

## Encoder Classes

Context Encoder

In [19]:
class XLNetContextEncoder(nn.Module):
    def __init__(self, xlnet_model, hidden_size, projection_dim):
        super().__init__()
        self.xlnet      = xlnet_model
        self.projection = nn.Sequential(
            nn.Linear(hidden_size, projection_dim),
            nn.LayerNorm(projection_dim),
            nn.GELU(),
        )

    def forward(self, input_ids, attention_mask, token_type_ids):
        outputs       = self.xlnet(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        hidden        = outputs.last_hidden_state
        mask_expanded = attention_mask.unsqueeze(-1).float()
        sum_hidden    = (hidden * mask_expanded).sum(dim=1)
        token_counts  = mask_expanded.sum(dim=1).clamp(min=1e-9)
        pooled        = sum_hidden / token_counts
        return self.projection(pooled)


class MultiContextXLNetEncoder(nn.Module):
    def __init__(self, model_name, projection_dim):
        super().__init__()
        print(f"Loading XLNet: {model_name} ...")
        self.xlnet_backbone   = XLNetModel.from_pretrained(model_name)
        hidden_size           = self.xlnet_backbone.config.d_model
        self.issue_encoder    = XLNetContextEncoder(self.xlnet_backbone, hidden_size, projection_dim)
        self.topic_encoder    = XLNetContextEncoder(self.xlnet_backbone, hidden_size, projection_dim)
        self.ideology_encoder = XLNetContextEncoder(self.xlnet_backbone, hidden_size, projection_dim)
        print(f"Model ready | hidden={hidden_size} | proj={projection_dim} | params={sum(p.numel() for p in self.parameters()):,}")

    def forward(self, batch):
        return {
            'issue_feature'    : self.issue_encoder(batch['issue_input_ids'],       batch['issue_attention_mask'],    batch['issue_token_type_ids']),
            'topic_feature'    : self.topic_encoder(batch['topic_input_ids'],       batch['topic_attention_mask'],    batch['topic_token_type_ids']),
            'ideology_feature' : self.ideology_encoder(batch['ideology_input_ids'], batch['ideology_attention_mask'], batch['ideology_token_type_ids']),
        }

## Building Model and Training

In [22]:
model = MultiContextXLNetEncoder(XLNET_MODEL_NAME, PROJECTION_DIM).to(DEVICE)
model.eval()

batch = next(iter(dataloader))
batch = {k: v.to(DEVICE) for k, v in batch.items()}

with torch.no_grad():
    features = model(batch)

for name, tensor in features.items():
    vec = tensor[0]
    print(f"{name:<22} | shape: {tuple(tensor.shape)} | norm: {vec.norm().item():.4f} | mean: {vec.mean().item():.4f}")

Loading XLNet: xlnet-base-cased ...


Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

XLNetModel LOAD REPORT from: xlnet-base-cased
Key            | Status     |  | 
---------------+------------+--+-
lm_loss.weight | UNEXPECTED |  | 
lm_loss.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model ready | hidden=768 | proj=256 | params=117,310,464
issue_feature          | shape: (16, 256) | norm: 10.1882 | mean: 0.3053
topic_feature          | shape: (16, 256) | norm: 9.9661 | mean: 0.3115
ideology_feature       | shape: (16, 256) | norm: 10.6631 | mean: 0.3030
